# LeetCode 2884 — Modify Columns bằng Python (pandas)

Link đề: https://leetcode.com/problems/modify-columns/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `employees` có cột `name`, `salary`; **nhân đôi toàn bộ cột `salary`** rồi trả bảng.
2. Hiểu **cột là một `Series`**, và phép toán số học trên Series (`* 2`) áp dụng cho **mọi phần tử cùng lúc** (vectorized).
3. Đi từ cách **vòng lặp tự nhân từng ô** → **`df["salary"] * 2`** → **`apply` / `map` với `lambda`** → **`assign`** → **`*=` và `mul`**.
4. Phân biệt **phép tính tạo Series mới** với **gán lại vào cột** (`df["salary"] = ...`): quên gán thì bảng không đổi.
5. Tránh các lỗi kinh điển: quên gán, nhầm `** 2` (bình phương) với `* 2`, `return` kết quả của câu lệnh gán, và hiểu vì sao hàm có thể sửa luôn bảng đầu vào.

## 2. Đề bài & ví dụ

Có một bảng `employees` với các cột `name` và `salary`. Công ty quyết định **tăng gấp đôi lương** cho mọi nhân viên.

Hãy viết hàm **nhân đôi cột `salary`** và trả về bảng đã sửa.

**Ví dụ**

- Input: DataFrame `employees`

| name | salary |
|---|---|
| Jack | 19666 |
| Piper | 74754 |
| Mia | 62509 |
| Ulysses | 54866 |

- Output:

| name | salary |
|---|---|
| Jack | 39332 |
| Piper | 149508 |
| Mia | 125018 |
| Ulysses | 109732 |

- Giải thích: mỗi giá trị trong cột `salary` được nhân với 2; cột `name` giữ nguyên.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "name": ["Jack", "Piper", "Mia", "Ulysses"],
    "salary": [19666, 74754, 62509, 54866],
})
print(employees)
print()
print(employees.dtypes)
print("pandas", pd.__version__)

      name  salary
0     Jack   19666
1    Piper   74754
2      Mia   62509
3  Ulysses   54866

name        str
salary    int64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Một cột là một `Series`

`employees["salary"]` lấy ra **một cột**. Cột đó là một `Series`: một dãy giá trị có nhãn (index) và có **kiểu dữ liệu** (`dtype`). Ở đây `salary` là số nguyên `int64`.

In [2]:
salary = employees["salary"]
print(type(salary))
print(salary)
print("dtype:", salary.dtype)
print("dạng list:", salary.tolist())

<class 'pandas.Series'>
0    19666
1    74754
2    62509
3    54866
Name: salary, dtype: int64
dtype: int64
dạng list: [19666, 74754, 62509, 54866]


### 3.2. Phép toán trên cả Series (vectorized)

Trong Python thuần, `[1, 2, 3] * 2` **lặp danh sách** (thành `[1, 2, 3, 1, 2, 3]`), không nhân từng số. Nhưng với `Series`, `s * 2` **nhân từng phần tử với 2** và trả về **Series mới**. Cách làm "áp dụng cho cả dãy trong một câu lệnh" gọi là **vectorized** (vector hóa): ngắn gọn và nhanh hơn vòng lặp Python rất nhiều.

In [3]:
print([1, 2, 3] * 2)                 # list: lặp lại danh sách
s = pd.Series([1, 2, 3])
print((s * 2).tolist())              # Series: nhân từng phần tử -> [2, 4, 6]
print((s + 10).tolist())             # cộng 10 vào từng phần tử
print((s ** 2).tolist())             # bình phương từng phần tử (khác * 2!)

[1, 2, 3, 1, 2, 3]
[2, 4, 6]
[11, 12, 13]
[1, 4, 9]


### 3.3. Phép tính KHÔNG tự đổi bảng — phải gán lại

`employees["salary"] * 2` chỉ **tạo ra một Series mới**; bảng `employees` vẫn nguyên. Muốn bảng đổi, phải **gán** kết quả vào cột: `employees["salary"] = employees["salary"] * 2`.

Hãy thử trên một bản sao (`copy()`) để không làm hỏng dữ liệu mẫu.
(Bản sao là một DataFrame mới; sửa nó không ảnh hưởng bản gốc.)

In [4]:
tmp = employees.copy()
_ = tmp["salary"] * 2                       # tính rồi vứt đi
print("sau khi chỉ tính:", tmp["salary"].tolist())     # chưa đổi

tmp["salary"] = tmp["salary"] * 2           # gán lại vào cột
print("sau khi gán    :", tmp["salary"].tolist())      # đã nhân đôi
print("bảng gốc       :", employees["salary"].tolist())  # không bị ảnh hưởng vì ta làm trên bản sao

sau khi chỉ tính: [19666, 74754, 62509, 54866]
sau khi gán    : [39332, 149508, 125018, 109732]
bảng gốc       : [19666, 74754, 62509, 54866]


### 3.4. `Series.apply(hàm)` và `lambda`

`s.apply(f)` gọi hàm `f` **cho từng phần tử** rồi gom kết quả thành Series mới. `lambda x: x * 2` là **hàm vô danh** viết trên một dòng: "nhận `x`, trả `x * 2`". `apply` chậm hơn `s * 2` (vì gọi hàm Python từng ô) nhưng linh hoạt khi phép biến đổi phức tạp (ví dụ rẽ nhánh `if`).

`Series.map(f)` tương tự cho trường hợp đơn giản.

In [5]:
s = pd.Series([1, 2, 3])
print(s.apply(lambda x: x * 2).tolist())     # [2, 4, 6]
print(s.map(lambda x: x * 2).tolist())       # [2, 4, 6]

def double(x):
    return x * 2

print(s.apply(double).tolist())              # có thể truyền hàm có tên

[2, 4, 6]
[2, 4, 6]
[2, 4, 6]


### 3.5. `assign` — tạo bảng mới có cột đã đổi

`df.assign(salary=...)` trả về **bảng mới** với cột `salary` thay bằng giá trị đưa vào, **không sửa `df` gốc**. Tham số có thể là Series hoặc một `lambda d: ...` nhận chính bảng. Rất hợp với kiểu viết nối nhiều bước (method chaining).

In [6]:
new_df = employees.assign(salary=lambda d: d["salary"] * 2)
print(new_df)
print("bảng gốc không đổi:", employees["salary"].tolist())

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732
bảng gốc không đổi: [19666, 74754, 62509, 54866]


### 3.6. `*=` và `mul`: nhân tại chỗ

- `df["salary"] *= 2` là **viết tắt** của `df["salary"] = df["salary"] * 2`;
- `s.mul(2)` là **tên hàm** của phép `s * 2` (có thể nối chuỗi `.mul(2).add(1)`).

In [7]:
tmp = employees.copy()
tmp["salary"] *= 2
print(tmp["salary"].tolist())

print(employees["salary"].mul(2).tolist())   # giống employees["salary"] * 2

[39332, 149508, 125018, 109732]
[39332, 149508, 125018, 109732]


### 3.7. Hàm có sửa bảng đầu vào không?

Trong Python, truyền DataFrame vào hàm là truyền **tham chiếu** tới cùng một đối tượng. Nếu trong hàm viết `employees["salary"] = ...` thì **bảng của người gọi cũng bị đổi** (vì cùng một đối tượng). LeetCode chỉ chấm giá trị trả về nên cách này vẫn được nhận. Nếu muốn hàm "sạch" không có tác dụng phụ, làm việc trên `copy()` hoặc dùng `assign`.

In [8]:
def double_inplace(df):
    df["salary"] = df["salary"] * 2        # sửa luôn bảng của người gọi
    return df

def double_pure(df):
    return df.assign(salary=df["salary"] * 2)    # tạo bảng mới

a = employees.copy()
r = double_inplace(a)
print("sau double_inplace: bảng truyền vào =", a["salary"].tolist(), "| cùng đối tượng?", r is a)

b = employees.copy()
r2 = double_pure(b)
print("sau double_pure   : bảng truyền vào =", b["salary"].tolist(), "| cùng đối tượng?", r2 is b)

sau double_inplace: bảng truyền vào = [39332, 149508, 125018, 109732] | cùng đối tượng? True
sau double_pure   : bảng truyền vào = [19666, 74754, 62509, 54866] | cùng đối tượng? False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một bảng lương trên giấy. Với mỗi dòng, **gạch số cũ, ghi số mới gấp đôi** ở cột lương; cột tên không động tới.

| nhãn | name | salary cũ | tính | salary mới |
|---|---|---|---|---|
| 0 | Jack | 19666 | 19666 × 2 | 39332 |
| 1 | Piper | 74754 | 74754 × 2 | 149508 |
| 2 | Mia | 62509 | 62509 × 2 | 125018 |
| 3 | Ulysses | 54866 | 54866 × 2 | 109732 |

Đây là thuật toán: **biến đổi từng ô của một cột bằng cùng một phép tính**. Có hai việc: (1) tính giá trị mới cho mọi ô, (2) **đặt chúng vào đúng cột**. Mọi cách ở dưới chỉ khác ở cách làm hai việc này.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

**Ý tưởng.** Duyệt từng giá trị lương, nhân 2, gom vào danh sách mới `new_salary`; cuối cùng **gán cả danh sách vào cột**. Danh sách phải **cùng độ dài** với bảng, nếu không pandas báo lỗi.

In [9]:
def modify_loop(employees: pd.DataFrame) -> pd.DataFrame:
    df = employees.copy()                      # không đụng bảng gốc
    new_salary = []
    for old in df["salary"]:                   # duyệt từng giá trị trong cột
        new_salary.append(old * 2)             # nhân đôi từng ô
    # gán vào cột; ghi rõ index và dtype để bảng rỗng vẫn giữ kiểu int64
    df["salary"] = pd.Series(new_salary, index=df.index, dtype=df["salary"].dtype)
    return df


print(modify_loop(employees))

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732


In [10]:
res = modify_loop(employees)
print("salary mới:", res["salary"].tolist())      # [39332, 149508, 125018, 109732]
print("dtype     :", res["salary"].dtype)         # int64: nhân số nguyên vẫn là số nguyên
print("name giữ nguyên:", res["name"].tolist() == employees["name"].tolist())

salary mới: [39332, 149508, 125018, 109732]
dtype     : int64
name giữ nguyên: True


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng. Vòng lặp `m` lần, mỗi lần `O(1)`: **Time `O(m)`**; danh sách `new_salary` có `m` phần tử (và bản sao bảng): **Space `O(m)`**.

**Nút thắt.** Vòng lặp Python chậm với bảng lớn, và dài dòng. pandas cho phép nhân cả cột một phát.

## 6. Ý tưởng thuật toán / bước đột phá

Một Series nhân với số sẽ **nhân từng phần tử** (vectorized). Vậy toàn bộ bài chỉ là:

`employees["salary"] = employees["salary"] * 2`

- vế phải: Series mới gồm lương nhân đôi;
- vế trái: ghi Series đó đè lên cột `salary` (các nhãn index khớp nhau nên ghép đúng hàng);
- nhớ `return employees` ở cuối.

Các cách khác: `apply`/`map` (Cách 3), `assign` (Cách 4), `*=`/`mul` (Cách 5).

## 7. Các cách tiếp theo

### Cách 2 — Nhân cả cột rồi gán lại (cách chuẩn)

**Giải thích.** `employees["salary"] * 2` tạo Series mới (vectorized); gán lại vào cột. Cách này sửa luôn `employees` đầu vào rồi trả về nó; LeetCode chấp nhận.

In [11]:
def modify_vectorized(employees: pd.DataFrame) -> pd.DataFrame:
    df = employees.copy()
    df["salary"] = df["salary"] * 2
    return df


print(modify_vectorized(employees))

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["salary"] = employees["salary"] * 2
    return employees

**Độ phức tạp Cách 2:** **Time `O(m)`** (một phép nhân trên `m` phần tử, thực hiện bằng mã C nên rất nhanh), **Space `O(m)`** (Series trung gian `m` phần tử).

### Cách 3 — `apply` / `map` với `lambda`

**Giải thích.** Gọi một hàm cho từng ô. Chậm hơn Cách 2 nhưng cần khi phép biến đổi không biểu diễn được bằng toán tử vector, ví dụ "nhân 2 nếu lương dưới 50000, ngược lại giữ nguyên".

In [13]:
def modify_apply(employees: pd.DataFrame) -> pd.DataFrame:
    df = employees.copy()
    df["salary"] = df["salary"].apply(lambda x: x * 2)
    return df


print(modify_apply(employees))
print(employees["salary"].map(lambda x: x * 2).tolist())     # map cho kết quả tương tự

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732
[39332, 149508, 125018, 109732]


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["salary"] = employees["salary"].apply(lambda x: x * 2)
    return employees

**Độ phức tạp Cách 3:** **Time `O(m)`** nhưng hằng số lớn (gọi hàm Python `m` lần), **Space `O(m)`**.

### Cách 4 — `assign`: tạo bảng mới (không sửa đầu vào)

**Giải thích.** `assign` trả về **bản mới** có cột `salary` đã đổi, nên bảng đầu vào giữ nguyên. Hợp với phong cách "hàm thuần" và method chaining.

In [15]:
def modify_assign(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.assign(salary=lambda d: d["salary"] * 2)


before = employees.copy()
print(modify_assign(employees))
pd.testing.assert_frame_equal(employees, before)
print("đầu vào không đổi")

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732
đầu vào không đổi


In [16]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.assign(salary=employees["salary"] * 2)

**Độ phức tạp Cách 4:** **Time `O(m)`**, **Space `O(m·c)`** (`assign` tạo bảng mới `c` cột; pandas có thể dùng chung bộ nhớ các cột không đổi nhờ copy-on-write, nhưng khi phân tích ta coi như bản sao).

### Cách 5 — `*=` hoặc `mul(2)`

**Giải thích.** `*=` nhân và gán trong một bước; `mul(2)` là dạng gọi hàm của phép nhân, có tham số `fill_value` hữu ích khi có giá trị thiếu.

In [17]:
def modify_imul(employees: pd.DataFrame) -> pd.DataFrame:
    df = employees.copy()
    df["salary"] *= 2
    return df


def modify_mul(employees: pd.DataFrame) -> pd.DataFrame:
    df = employees.copy()
    df["salary"] = df["salary"].mul(2)
    return df


print(modify_imul(employees)["salary"].tolist())
print(modify_mul(employees)["salary"].tolist())

[39332, 149508, 125018, 109732]
[39332, 149508, 125018, 109732]


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`**.

> Cách 2, 4, 5 đều được nhận. Ngắn nhất và dễ đọc nhất: `employees["salary"] *= 2`, hoặc dạng đầy đủ ở Cách 2.

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết, in từng ô; (b) pipeline pandas `tính Series mới → gán → kiểm tra`.

In [18]:
def trace_loop(employees):
    df = employees.copy()
    new_salary = []
    for step, (label, name, old) in enumerate(zip(df.index, df["name"], df["salary"]), start=1):
        new = old * 2
        print(f"bước {step}: nhãn={label}, name={name!r}, salary cũ={old}, salary mới={old}*2={new}")
        new_salary.append(new)
        print(f"          new_salary = {new_salary}")
    df["salary"] = new_salary
    print("\ngán danh sách vào cột salary ->", df["salary"].tolist())
    return df


_ = trace_loop(employees)

bước 1: nhãn=0, name='Jack', salary cũ=19666, salary mới=19666*2=39332
          new_salary = [39332]
bước 2: nhãn=1, name='Piper', salary cũ=74754, salary mới=74754*2=149508
          new_salary = [39332, 149508]
bước 3: nhãn=2, name='Mia', salary cũ=62509, salary mới=62509*2=125018
          new_salary = [39332, 149508, 125018]
bước 4: nhãn=3, name='Ulysses', salary cũ=54866, salary mới=54866*2=109732
          new_salary = [39332, 149508, 125018, 109732]

gán danh sách vào cột salary -> [39332, 149508, 125018, 109732]


In [19]:
def trace_pandas(employees):
    df = employees.copy()
    print("BƯỚC 0 - bảng gốc", df.shape, "| dtype salary:", df["salary"].dtype)
    print(df)

    doubled = df["salary"] * 2
    print("\nBƯỚC 1 - df['salary'] * 2 (Series mới, bảng CHƯA đổi):")
    print(doubled.tolist())
    print("salary trong bảng lúc này:", df["salary"].tolist())

    df["salary"] = doubled
    print("\nBƯỚC 2 - df['salary'] = doubled (bảng đã đổi):")
    print(df)

    same = df.equals(employees.assign(salary=employees["salary"] * 2))
    print("\ngiống assign(salary=salary*2)?", same)
    return df


_ = trace_pandas(employees)

BƯỚC 0 - bảng gốc (4, 2) | dtype salary: int64
      name  salary
0     Jack   19666
1    Piper   74754
2      Mia   62509
3  Ulysses   54866

BƯỚC 1 - df['salary'] * 2 (Series mới, bảng CHƯA đổi):
[39332, 149508, 125018, 109732]
salary trong bảng lúc này: [19666, 74754, 62509, 54866]

BƯỚC 2 - df['salary'] = doubled (bảng đã đổi):
      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732

giống assign(salary=salary*2)? True


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | cột `salary` | `[19666, 74754, 62509, 54866]` |
| 1 | `salary * 2` | Series `[39332, 149508, 125018, 109732]` (bảng chưa đổi) |
| 2 | gán vào cột | bảng có `salary` mới, `name` không đổi |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [20]:
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["salary"] = employees["salary"] * 2
    return employees


demo = employees.copy()          # dùng bản sao để các cell sau còn dữ liệu gốc
print(modifySalaryColumn(demo))

      name  salary
0     Jack   39332
1    Piper  149508
2      Mia  125018
3  Ulysses  109732


## 10. Vì sao thuật toán đúng

- **Đúng phép tính:** `Series * 2` nhân **từng** phần tử với 2, đúng điều đề yêu cầu "gấp đôi lương".
- **Đúng vị trí:** phép gán `df["salary"] = Series` ghép theo **nhãn index**. Series được tính từ chính cột đó nên nhãn giống hệt, mỗi lương mới nằm đúng hàng của nhân viên đó.
- **Cột khác không đổi:** ta chỉ gán vào cột `salary`; `name` giữ nguyên, thứ tự hàng và index cũng giữ nguyên.
- **Kiểu dữ liệu:** `int64 * 2` vẫn là `int64` (trong phạm vi đề, không tràn số), nên giá trị là số nguyên đúng như ví dụ.
- **Tính chất kiểm được:** `new.sum() == 2 * old.sum()` và `new / 2 == old` với mọi hàng.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

- **Time:** `O(m)`: mỗi ô của cột `salary` được nhân đúng một lần. Vectorized chạy mã C nên hằng số nhỏ; vòng lặp Python/`apply` hằng số lớn hơn nhưng cùng bậc.
- **Space:** `O(m)` cho Series tạm; nếu tạo bảng mới (`assign`, `copy`) thì `O(m·c)`.
- Không thể nhanh hơn `O(m)` vì phải chạm vào mọi ô lương ít nhất một lần.

## 12. Bộ test

Ta đưa các cách vào một từ điển `impls`; mỗi lần gọi đều truyền **bản sao** của bảng để các hàm sửa tại chỗ không làm hỏng dữ liệu của test sau.

In [21]:
impls = {
    "loop": modify_loop,
    "vectorized": modify_vectorized,
    "apply": modify_apply,
    "assign": modify_assign,
    "imul": modify_imul,
    "mul": modify_mul,
    "submit": modifySalaryColumn,
}

expected_example = pd.DataFrame({
    "name": ["Jack", "Piper", "Mia", "Ulysses"],
    "salary": [39332, 149508, 125018, 109732],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(employees.copy()), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [22]:
def make(salaries, names=None):
    n = len(salaries)
    return pd.DataFrame({"name": names if names is not None else [f"e{i}" for i in range(n)], "salary": salaries})


def check_all(df, expected_salaries, label):
    for key, fn in impls.items():
        got = fn(df.copy())["salary"].tolist()
        assert got == expected_salaries, (label, key, got, expected_salaries)
    print("PASS", label, "->", expected_salaries)


check_all(make([1]), [2], "một hàng")
check_all(make([0, 0, 0]), [0, 0, 0], "toàn lương 0")
check_all(make([5, 5, 5]), [10, 10, 10], "lương bằng nhau")
check_all(make([10**9, 2 * 10**9]), [2 * 10**9, 4 * 10**9], "số lớn (int64 vẫn đủ)")
check_all(make([-3, 4]), [-6, 8], "số âm")

PASS một hàng -> [2]
PASS toàn lương 0 -> [0, 0, 0]
PASS lương bằng nhau -> [10, 10, 10]
PASS số lớn (int64 vẫn đủ) -> [2000000000, 4000000000]
PASS số âm -> [-6, 8]


### 12.2. Bảng rỗng, tên cột/giá trị thiếu, kiểu thực, đầu vào, và phản ví dụ

In [23]:
# Bảng rỗng
empty = pd.DataFrame({"name": pd.Series([], dtype="str"), "salary": pd.Series([], dtype="int64")})
for key, fn in impls.items():
    out = fn(empty.copy())
    assert len(out) == 0 and list(out.columns) == ["name", "salary"], key
print("PASS bảng rỗng")

# Lương kiểu thực và có giá trị thiếu: NaN * 2 vẫn là NaN
f = pd.DataFrame({"name": ["a", "b"], "salary": [1.5, np.nan]})
res = modifySalaryColumn(f.copy())
assert res["salary"].iloc[0] == 3.0 and pd.isna(res["salary"].iloc[1])
print("PASS lương thực + NaN:", res["salary"].tolist())

# Các cách "thuần" (không sửa đầu vào)
before = employees.copy()
for key in ["loop", "vectorized", "apply", "assign", "imul", "mul"]:
    impls[key](employees)
pd.testing.assert_frame_equal(employees, before)
print("PASS các cách thuần không sửa đầu vào")

# Tính chất: tổng gấp đôi, cột name không đổi, thứ tự hàng không đổi
out = modifySalaryColumn(employees.copy())
assert out["salary"].sum() == 2 * employees["salary"].sum()
assert out["name"].tolist() == employees["name"].tolist()
assert out.index.tolist() == employees.index.tolist()
print("PASS tính chất kết quả")

# Phản ví dụ: ** 2 là bình phương, không phải gấp đôi
wrong = employees["salary"] ** 2
assert wrong.tolist() != (employees["salary"] * 2).tolist()
print("PASS phản ví dụ: ** 2 cho", wrong.tolist()[:2], "...")

PASS bảng rỗng
PASS lương thực + NaN: [3.0, nan]
PASS các cách thuần không sửa đầu vào
PASS tính chất kết quả
PASS phản ví dụ: ** 2 cho [386751556, 5588160516] ...


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [24]:
import random

random.seed(2884)
for trial in range(200):
    n = random.randint(0, 20)
    sal = [random.randint(-1000, 10**6) for _ in range(n)]
    df = pd.DataFrame({"name": pd.Series([f"n{i}" for i in range(n)], dtype="str"), "salary": pd.Series(sal, dtype="int64")})
    ref = pd.DataFrame({"name": df["name"], "salary": pd.Series([2 * v for v in sal], dtype="int64")})
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df.copy()), ref)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính xong nhưng quên gán lại

In [25]:
def sai_quen_gan(employees):
    employees["salary"] * 2          # tạo Series rồi vứt đi
    return employees


tmp = employees.copy()
print("sai  :", sai_quen_gan(tmp)["salary"].tolist())      # vẫn là lương cũ
print("đúng :", modifySalaryColumn(employees.copy())["salary"].tolist())

sai  : [19666, 74754, 62509, 54866]
đúng : [39332, 149508, 125018, 109732]


**Vì sao sai:** `employees["salary"] * 2` chỉ **tính** ra Series mới; không câu lệnh nào ghi nó vào bảng. Phải có phép gán `employees["salary"] = ...` (hoặc `*=`).

### Lỗi 2: Nhầm `** 2` (bình phương) với `* 2` (gấp đôi)

In [26]:
sai = employees["salary"] ** 2
dung = employees["salary"] * 2
print("sai  :", sai.tolist()[:2], "...")     # 19666**2 = 386751556, ...
print("đúng :", dung.tolist()[:2], "...")

sai  : [386751556, 5588160516] ...
đúng : [39332, 149508] ...


**Vì sao sai:** `**` là **lũy thừa**. Đề yêu cầu "gấp đôi" nên dùng phép nhân `* 2`. Các lỗi đánh máy kiểu này rất khó thấy khi chỉ nhìn code, nên hãy luôn chạy ví dụ của đề và so sánh.

### Lỗi 3: `return` kết quả của câu lệnh gán / viết sai kiểu trả về

In [27]:
def sai_return_series(employees):
    return employees["salary"] * 2            # trả về Series, không phải DataFrame


r = sai_return_series(employees)
print("sai  : kiểu trả về =", type(r).__name__)
print("đúng : kiểu trả về =", type(modifySalaryColumn(employees.copy())).__name__)

sai  : kiểu trả về = Series
đúng : kiểu trả về = DataFrame


**Vì sao sai:** đề yêu cầu trả **cả bảng** (`name` và `salary`), tức một `DataFrame`. Trả Series làm mất cột `name`. Ngoài ra, một câu lệnh gán như `df["salary"] = ...` **không có giá trị trả về**; không thể viết `return df["salary"] = ...` (lỗi cú pháp).

### Lỗi 4: Gán danh sách có độ dài sai

In [28]:
tmp = employees.copy()
try:
    tmp["salary"] = [1, 2, 3]            # bảng có 4 hàng nhưng danh sách chỉ 3 phần tử
except ValueError as e:
    print("ValueError:", e)

ValueError: Length of values (3) does not match length of index (4)


**Vì sao sai:** khi gán **list** vào cột, độ dài phải bằng số hàng. Với cách vòng lặp (Cách 1) hãy kiểm tra `len(new_salary) == len(df)`; cách vectorized không gặp lỗi này vì Series tính ra luôn cùng độ dài.

### Lỗi 5: Dùng `apply` trên cả bảng thay vì một cột

In [29]:
wrong = employees.copy()
try:
    wrong = wrong.apply(lambda x: x * 2)         # áp dụng lên MỌI cột, kể cả name
    print(wrong)
except Exception as e:
    print("lỗi:", type(e).__name__)
print("đúng: chỉ nhân cột salary ->", employees.assign(salary=employees["salary"] * 2)["name"].tolist())

             name  salary
0        JackJack   39332
1      PiperPiper  149508
2          MiaMia  125018
3  UlyssesUlysses  109732
đúng: chỉ nhân cột salary -> ['Jack', 'Piper', 'Mia', 'Ulysses']


**Vì sao sai:** `df.apply(f)` gọi `f` cho **từng cột** (cả `name`); nhân chuỗi với 2 trong Python thành **lặp chuỗi** ("Jack" -> "JackJack"), làm hỏng cột `name`. Luôn chọn rõ cột: `df["salary"].apply(...)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đổi / tăng / nhân / cộng cho cả một cột" → nghĩ tới **phép toán vector** `df["col"] * k` rồi **gán lại**.
- Công thức nhớ: **`df["col"] = <biểu thức trên df["col"]>`**; hoặc dùng `df["col"] *= k`.
- Muốn **không sửa bảng đầu vào**: `df.assign(col=...)` hoặc `df = df.copy()` trước.
- Phép biến đổi phức tạp có `if`: dùng `np.where(điều_kiện, a, b)` (vectorized) hoặc `apply` (linh hoạt nhưng chậm).
- Kiểm tra nhanh bằng `df["col"].sum()` hoặc `df.head()` trước và sau.
- Tên hàm trên LeetCode phải khớp đúng (`modifySalaryColumn`); sai tên là lỗi dù logic đúng.

## 15. Biến thể của bài

1. Tăng lương **10%**: `df["salary"] = df["salary"] * 1.1` (kết quả thành số thực).
2. Chỉ nhân đôi lương nhân viên **dưới 50000**: `np.where(df["salary"] < 50000, df["salary"] * 2, df["salary"])`.
3. Thêm cột mới `bonus = salary * 0.1` mà không đổi `salary`: `df["bonus"] = df["salary"] * 0.1`.
4. Nhân đôi **nhiều cột** cùng lúc: `df[["salary", "bonus"]] *= 2`.
5. Giới hạn trần lương: `df["salary"].clip(upper=100000)`.
6. Đổi thành lương **theo tháng**: `df["salary"] / 12` (và làm tròn `round(2)`).

Ta thử vài biến thể:

In [30]:
v = employees.copy()
print("1. +10%      :", (v["salary"] * 1.1).round(1).tolist())
print("2. chỉ <50000:", np.where(v["salary"] < 50000, v["salary"] * 2, v["salary"]).tolist())
v["bonus"] = v["salary"] * 0.1
print("3. cột bonus :", v["bonus"].round(1).tolist())
print("5. clip trần :", v["salary"].clip(upper=60000).tolist())

1. +10%      : [21632.6, 82229.4, 68759.9, 60352.6]
2. chỉ <50000: [39332, 74754, 62509, 54866]
3. cột bonus : [1966.6, 7475.4, 6250.9, 5486.6]
5. clip trần : [19666, 60000, 60000, 54866]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + list | nhân từng ô rồi gán list | `O(m)` (Python, chậm) | `O(m)` | hiểu bản chất; không dùng trên dữ liệu lớn |
| 2. `df["salary"] = df["salary"] * 2` | vectorized + gán | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `apply(lambda x: x*2)` | gọi hàm từng ô | `O(m)` (hằng số lớn) | `O(m)` | phép biến đổi có rẽ nhánh phức tạp |
| 4. `assign(salary=...)` | bảng mới, không sửa đầu vào | `O(m)` | `O(m·c)` | method chaining, hàm thuần |
| 5. `*=` / `mul(2)` | nhân + gán gọn | `O(m)` | `O(m)` | ngắn gọn; `mul` có `fill_value` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[3, 4] * 2`, `(pd.Series([3, 4]) * 2).tolist()`, `(pd.Series([3, 4]) ** 2).tolist()`, `employees["salary"].dtype`, `type(employees["salary"] * 2).__name__`.

**Bài 2 (tính tay).** Với lương `[100, 250, 0, 75]`: tính lương sau khi nhân đôi; tính tổng trước và sau; nếu chỉ nhân đôi lương `< 100` thì ra gì?

**Bài 3 (điền TODO).** Hoàn thành `modify_todo(df)` (không sửa bảng đầu vào).

In [31]:
def modify_todo(df):
    # TODO 1: out = df.copy()
    # TODO 2: out["salary"] = out["salary"] * 2
    # TODO 3: return out
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `modify_loop2(employees)` bằng vòng lặp thuần Python (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `raise_low_salaries(df, threshold)`: nhân đôi lương của những người có lương `< threshold`, giữ nguyên người còn lại. (b) Viết `add_bonus(df, rate)`: thêm cột `total = salary * (1 + rate)` mà không làm đổi cột `salary` và không sửa bảng đầu vào.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[3, 4, 3, 4]`, `[6, 8]`, `[9, 16]`, `int64`, `Series`. Bài 2: nhân từng số cho 2, cộng lại.
- **Mức 2 (vừa):** Bài 2: `[200, 500, 0, 150]`; tổng trước `425`, sau `850`; chỉ nhân đôi số `< 100` -> `[100, 250, 0, 150]`. Bài 3: ba dòng đã ghi trong comment. Bài 4: tạo danh sách mới, duyệt từng giá trị, nhân 2, cuối cùng gán list vào cột.
- **Mức 3 (sâu):** Bài 5(a): `np.where(df["salary"] < threshold, df["salary"] * 2, df["salary"])` hoặc `df["salary"].where(df["salary"] >= threshold, df["salary"] * 2)`. Bài 5(b): `df.assign(total=df["salary"] * (1 + rate))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert [3, 4] * 2 == [3, 4, 3, 4]
assert (pd.Series([3, 4]) * 2).tolist() == [6, 8]
assert (pd.Series([3, 4]) ** 2).tolist() == [9, 16]
assert str(employees["salary"].dtype) == "int64"
assert type(employees["salary"] * 2).__name__ == "Series"

# Bài 2
sal = pd.Series([100, 250, 0, 75])
assert (sal * 2).tolist() == [200, 500, 0, 150]
assert sal.sum() == 425 and (sal * 2).sum() == 850
assert np.where(sal < 100, sal * 2, sal).tolist() == [100, 250, 0, 150]

# Bài 3
def modify_todo_ans(df):
    out = df.copy()
    out["salary"] = out["salary"] * 2
    return out

before = employees.copy()
pd.testing.assert_frame_equal(modify_todo_ans(employees), expected_example)
pd.testing.assert_frame_equal(employees, before)      # bảng đầu vào không đổi

# Bài 4
def modify_loop2(employees):
    df = employees.copy()
    new_salary = []
    for old in df["salary"]:
        new_salary.append(old * 2)
    df["salary"] = new_salary
    return df

pd.testing.assert_frame_equal(modify_loop2(employees), expected_example)

# Bài 5(a)
def raise_low_salaries(df, threshold):
    out = df.copy()
    out["salary"] = np.where(out["salary"] < threshold, out["salary"] * 2, out["salary"])
    return out

assert raise_low_salaries(employees, 55000)["salary"].tolist() == [39332, 74754, 62509, 109732]

# Bài 5(b)
def add_bonus(df, rate):
    return df.assign(total=df["salary"] * (1 + rate))

r = add_bonus(employees, 0.5)
assert r["total"].tolist() == [29499.0, 112131.0, 93763.5, 82299.0]
assert r["salary"].tolist() == employees["salary"].tolist() and "total" not in employees.columns
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi một cột bằng phép tính số học)

```python
import pandas as pd

def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["salary"] = employees["salary"] * 2
    return employees
```

**Checklist**

- [ ] Có **phép gán** `employees["salary"] = ...` (không chỉ tính rồi bỏ)?
- [ ] Phép tính là `* 2` (gấp đôi), không phải `** 2` hay `+ 2`?
- [ ] **Trả về** DataFrame (không trả Series, không trả kết quả của câu lệnh gán)?
- [ ] Chỉ động tới cột `salary`; cột `name` và thứ tự hàng giữ nguyên?
- [ ] Biết hàm này sửa luôn bảng đầu vào; nếu cần thuần thì dùng `assign` / `copy()`?
- [ ] Đã thử bảng rỗng, lương 0, lương âm / số lớn?
- [ ] Skeleton đúng `def modifySalaryColumn(employees: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?